<a href="https://colab.research.google.com/github/marcory-hub/yolo11n-on-grove-vision-ai-v2/blob/main/YOLO_best_pt_naar_full_int_quant_vela_tflite_2026_03_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

Last accessed:2026-03-11
yolo26n 224 best no_post=True

tested nms=True or end2end=None both not valid in https://github.com/kris-himax/ultralytics

dataset
- images
  - train
  - val
- labels
  - train
  - val

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy zipped dataset to colab
!cp '/content/drive/MyDrive/dataset_images_labels.zip' '/content/dataset.zip'

# Unzip quietly to avoid massive logs (remove -q -n flags if needed)
# -q = quiet (no file list output)
# -n = no overwrite (skips existing files)
!unzip -nq '/content/dataset.zip' -d '/content/dataset/'

In [ ]:
# Copy model weights from google drive
!cp '/content/drive/MyDrive/best.pt' '/content/best.pt'

In [ ]:
# Make a calibrationset with seed 42
import os
import random
import shutil

# Configuration
nc = 4
names_list = ['amel', 'vcra', 'vespsp', 'vvel']
num_images = 500

# NEW STRUCTURE PATHS
# Root of your dataset (e.g., /content/dataset)
dataset_root = "/content/dataset"
temp_dir = "/content/temp_subset"

# Set seed for reproducibility
random.seed(42)

# Clean up existing temp folder
if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
    print(f"Cleaned up existing directory: {temp_dir}")

def copy_random_subset(split_name, dst_root, num_images):
    """
    split_name: 'train' or 'val'
    dataset_root: The folder containing 'images' and 'labels' folders
    """
    src_img_path = os.path.join(dataset_root, "images", split_name)
    src_lbl_path = os.path.join(dataset_root, "labels", split_name)

    if not os.path.exists(src_img_path):
        print(f"Warning: {src_img_path} not found. Skipping.")
        return 0

    all_images = [f for f in os.listdir(src_img_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]

    # Randomly sample
    selected = random.sample(all_images, min(num_images, len(all_images)))

    # Create destination directories
    dst_img_dir = os.path.join(dst_root, "images", split_name)
    dst_lbl_dir = os.path.join(dst_root, "labels", split_name)
    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    count = 0
    for img in selected:
        label = os.path.splitext(img)[0] + '.txt'

        # Copy image
        shutil.copy(os.path.join(src_img_path, img), os.path.join(dst_img_dir, img))

        # Copy label
        label_file_path = os.path.join(src_lbl_path, label)
        if os.path.exists(label_file_path):
            shutil.copy(label_file_path, os.path.join(dst_lbl_dir, label))

        count += 1
    return count

# Execute copying for both splits
train_count = copy_random_subset("train", temp_dir, num_images)
val_count = copy_random_subset("val", temp_dir, num_images)

# Prepare YAML data
# Note: YOLO usually expects paths relative to the 'path' key or absolute paths
temp_data_string = f"""
path: {temp_dir}
train: images/train
val: images/val
nc: {nc}
names: {names_list}
"""

# Write the new YAML file
with open("/content/temp_data.yaml", 'w') as f:
    f.write(temp_data_string.strip())

print("--- Summary ---")
print(f"Train images sampled: {train_count}")
print(f"Val images sampled: {val_count}")
print(f"YAML created at: /content/temp_data.yaml")

In [ ]:
!git clone https://github.com/kris-himax/ultralytics
%cd ultralytics
%pip install .
%cd ..
import ultralytics
ultralytics.checks()

# 0.a export original yolo11n int8 tflite with BatchMatMul Operater
no_post=False

PyTorch: starting from 'best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 8, 1029) (5.2 MB)

imgsz=224

In [ ]:
#!yolo export model=best.pt format=tflite int8=True  nms=False no_post=False imgsz=224 data=/content/temp_data.yaml

# 0.b export yolo11n int8 tflite with BatchMatMul Operater without post-processing
no_post=True

PyTorch: starting from 'best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) ((1, 68, 28, 28), (1, 68, 14, 14), (1, 68, 7, 7)) (5.2 MB)




In [ ]:
!yolo export model=best.pt format=tflite int8=True end2end=None no_post=True imgsz=224 data=/content/temp_data.yaml

# 0.1 install vela compiler

In [ ]:
!pip3 install ethos-u-vela
!wget https://raw.githubusercontent.com/HimaxWiseEyePlus/ML_FVP_EVALUATION/main/vela/himax_vela.ini

# 0.2 compile yolo11 int8 tflite to vela model

In [ ]:
!vela --accelerator-config ethos-u55-64 --config himax_vela.ini --system-config My_Sys_Cfg --memory-mode My_Mem_Mode_Parent --output-dir ./ ./best_saved_model/best_full_integer_quant.tflite

In [ ]:
from google.colab import files

!cp /content/best_full_integer_quant_vela.tflite /content/best_saved_model
!zip -r /content/yolo26n_vespa_2026-02v1_allpx_224_full_integer_quant_vela_tflite_nopost_noend2end.zip /content/best_saved_model
files.download('/content/yolo26n_vespa_2026-02v1_allpx_224_full_integer_quant_vela_tflite_nopost_noend2end.zip')

In [ ]:
# pt vs int8 comparison, do not run this with no_post int8
import os
import pandas as pd
from ultralytics import YOLO

# 1. Configuration
pt_path = '/content/best.pt'
tflite_path = '/content/best_saved_model/best_full_integer_quant.tflite'
save_dir = '/content/best_saved_model'
output_file = os.path.join(save_dir, 'detailed_model_comparison.txt')

# 2. Run Validation
# Ensuring imgsz matches your training size
model_pt = YOLO(pt_path)
model_tflite = YOLO(tflite_path, task='detect')

print("Validating PyTorch model...")
res_pt = model_pt.val(data='/content/dataset/data.yaml', imgsz=224, conf=0.001, verbose=False)

print("Validating TFLite (INT8) model...")
res_tflite = model_tflite.val(data='/content/dataset/data.yaml', imgsz=224, conf=0.001, verbose=False)

# 3. Extract Metrics for the first 4 classes
class_names = list(model_pt.names.values())[:4]

report_df = pd.DataFrame({
    "Class": class_names,
    "PT_mAP50": res_pt.box.ap50[:4],
    "TFLite_mAP50": res_tflite.box.ap50[:4],
    "PT_mAP50-95": res_pt.box.maps[:4],
    "TFLite_mAP50-95": res_tflite.box.maps[:4]
})

# Calculate Deltas
report_df['Delta_mAP50'] = report_df['TFLite_mAP50'] - report_df['PT_mAP50']
report_df['Delta_mAP50-95'] = report_df['TFLite_mAP50-95'] - report_df['PT_mAP50-95']

# 4. Generate Comprehensive Report String
report_content = f"""=====================================================
YOLO26 PyTorch vs TFLite (INT8) Detailed Comparison
=====================================================

PER-CLASS METRICS:
{report_df.to_string(index=False, float_format="%.4f")}

OVERALL SUMMARY:
Metric          | PyTorch | TFLite  | Delta
----------------|---------|---------|--------
mAP50           | {res_pt.box.map50:.4f}  | {res_tflite.box.map50:.4f}  | {res_tflite.box.map50 - res_pt.box.map50:.4f}
mAP50-95 (mean) | {res_pt.box.map:.4f}  | {res_tflite.box.map:.4f}  | {res_tflite.box.map - res_pt.box.map:.4f}

Validation Settings:
- Image Size: 224
- Confidence: 0.001
- Quantization: Full Integer (INT8)
"""

# 5. Save and Display
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

with open(output_file, 'w') as f:
    f.write(report_content)

print(f"\nReport successfully saved to: {output_file}")
print("\n" + report_content)